In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize

# Configuration
DATA_DIR = Path.cwd() / "testfiles_" / "data"
CSV_PATH = DATA_DIR / "test5_2.csv"

# Load covariance matrix
covar_data = pd.read_csv(CSV_PATH, header=0)
covar_matrix = covar_data.values

def calculate_portfolio_vol(w, cov):
    """Calculate portfolio volatility"""
    return np.sqrt(np.dot(w, np.dot(cov, w)))

def calculate_risk_components(w, cov):
    """Calculate risk contribution of each asset"""
    port_vol = calculate_portfolio_vol(w, cov)
    marginal_risk = np.dot(cov, w)
    risk_components = w * marginal_risk / port_vol
    return risk_components

def risk_parity_objective_func(w, cov):
    """Objective: minimize variance of risk contributions"""
    risk_comp = calculate_risk_components(w, cov)
    avg_risk = np.mean(risk_comp)
    deviations = risk_comp - avg_risk
    return np.sum(deviations ** 2) * 1e5

# Setup optimization
num_assets = covar_matrix.shape[0]
init_w = np.ones(num_assets) / num_assets

constraint = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}
weight_bounds = tuple((0, None) for _ in range(num_assets))

# Run optimization
optimization_result = minimize(
    risk_parity_objective_func,
    init_w,
    args=(covar_matrix,),
    method='SLSQP',
    bounds=weight_bounds,
    constraints=constraint,
    options={'ftol': 1e-9, 'maxiter': 1000}
)

# Extract and normalize weights
final_weights = optimization_result.x
final_weights = final_weights / np.sum(final_weights)

# Output
print('w')
for weight in final_weights:
    print(weight)

w
0.03546371505558438
0.025806189300902053
0.05646929004968188
0.265920158838064
0.6163406467557677
